In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

# 백터DB:Chroma VS Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

# 1. knowlege base 구성을 위한 데이터 생성

In [2]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("./tex_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문서를 쪼갤때 1500글자씩 chunking
    chunk_overlap=200, # 200문자씩 overlap 하여 chunking
    # separators=["\n\n","\n"," ",""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n","\n"," ",""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 3.75 s
Wall time: 4.12 s


195

In [10]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(
    #api_key = os.getenv('OPENAI_API_KEY'),
    model = 'solar-embedding-1-large-passage'
)

In [11]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(
    #api_key = os.getenv('OPENAI_API_KEY'),
    model = 'text-embedding-3-large'
)

In [12]:
len(embedding.embed_query("샘플"))

3072

In [ ]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [15]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    #api_key=os.getenv(PINECONE_API_KEY)
)
# 데이터를 처음 저장할때
index_name = 'tax-index-openai'
database = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

# 업로드된 백터DB를 load할때
# database = PineconeVectorStore(
#     embedding=embedding,
#     index_name=index_name
# )

CPU times: total: 3.25 s
Wall time: 8.73 s


# 2. 답변생성을 위한 Retrieval

In [16]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query,k=3)


In [18]:
retrieved_doc="\n--\n".join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])
# print(retrieved_doc)

# 3. 답변생성

In [19]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [ ]:
# from langchain_upstage import ChatOpenAI
# llm = ChatOpenAI(model='solar-pro2')

In [20]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가이다. [context]를 참고해서 사용자 질문에 답하세요.[context]는 다음과 같아요
{retrieved_doc}
- 질문 :{query}"""

In [21]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉이 5천만원인 직장인의 소득세액을 계산하기 위해서는 다음과 같은 절차를 따릅니다.

1. **과세표준 산정**  
근로소득에 대한 기본 공제, 근로소득공제, 기타 공제 등을 고려한 후 과세표준을 산정합니다.

2. **근로소득공제**  
2023년 기준으로, 연봉 5,000만원인 근로자의 근로소득공제는 대략 약 1,540만원 내지 1,560만원 정도로 예상됩니다. 정확한 금액은 소득공제 표와 세법 개정에 따라 상이할 수 있으나, 대표적으로 아래 재정표를 참고합니다.

예를 들어, 연봉 5천만원의 경우  
근로소득공제 = 연봉 × 일정 비율 + 일정 공제액 (구체적인 계산은 공제표 참조)

일반적으로 5천만원의 근로소득공제는 약 1,560만원 정도입니다.

3. **과세표준 계산**  
연봉 5,000만원 - 근로소득공제 약 1,560만원 = 약 3,440만원

4. **세율 적용**  
한국 소득세율은 구간별 누진세율이 적용됩니다.  
2023년 기준 세율 구간은 다음과 같으며,

| 과세표준 구간 (원) | 세율 | 누진공제액(원) | 세액 계산 공식                       |
|-----------------------|-------|------------------|-------------------------------------|
| 1,200만원 이하        | 6%    | 0                | 과세표준 × 6%                        |
| 1,200만원 초과 ~ 4,600만원 이하 | 15%  | 108만원          | (과세표준 × 15%) - 108만원           |

계산  
과세표준 ≈ 3,440만원

세액 = (3,440만원 × 15%) - 108만원  
= (34,400,000 × 0.15) - 1,080,000  
= 5,160,000 - 1,080,000  
= 약 4,080,000원

5. **종합적 세액**  
이것이 2023년 당해 연도 예상 소득세입니다.